# 04 — Локальная оценка и submission

## tl;dr
evaluation_model получил 66.40 с MAE на 473 точках повторного локального audit; 382 точки ранее встречались в audit artifacts, поэтому это не unseen. Отдельный submission_model переобучил выбранный регрессор на всех 4 434 train labels и записал 151 prediction. submission.csv перечитан с диска: schema, порядок, покрытие и конечность верны. Platform score отсутствует.


## Context & Methods
Воспроизведение из корня: .venv/bin/python -m transport_ml.predict --data-dir data --model-dir artifacts/t3-full-received-v2/submission_model --output submission.csv --report .tasks/T-3-2026-09-25-ml-submission/artifacts/submission_validation.json. Audit metrics принадлежат evaluation_model; submission_model refit на всех train labels не имеет локального audit score. Inference читает validate/points.csv, validate/traffic.csv и validate/schedule_plan.csv. Test schedule facts и реконструируемый validate proxy не используются.


In [1]:
from pathlib import Path
import sys
import os
import json
import pandas as pd
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
from transport_ml.predict import validate_submission
model_dir = root / os.environ.get('TRANSPORT_MODEL_RUN', 'artifacts/t3-full-received-v2') / 'evaluation_model'
metrics = json.loads((model_dir / 'metrics.json').read_text())
report = json.loads((root / '.tasks/T-3-2026-09-25-ml-submission/artifacts/submission_validation.json').read_text())
template = pd.read_csv(root / 'data/sample_submission.csv', sep=';', dtype={'sample_id': str})
readback = validate_submission(root / 'submission.csv', template.sample_id)
assert readback['rows'] == report['rows'] == 151
assert report['audit_exposure']['current_audit_points'] == metrics['split']['audit']['n']
print('Previously exposed audit rows:', report['audit_exposure']['previously_exposed_current_audit_points'])
assert len(report['model_file_manifest']) == 4
display(pd.DataFrame([readback]))


Previously exposed audit rows: 382


,rows,columns,unique_ids,finite_predictions,template_order
0,151,"[sample_id, prediction]",151,151,True


## Data & Results
Audit — временной отрезок train labels. Test — ранее просмотренный набор того же дня; оба являются локальными оценками.


In [2]:
rows = [{'set': 'chronological_audit', 'mae_s': metrics['chronological_audit']['regression']['mae_s'], 'n': metrics['chronological_audit']['regression']['n']}]
if 'official_test_same_day_diagnostic' in metrics:
    value = metrics['official_test_same_day_diagnostic']['regression']
    rows.append({'set': 'exposed_test_diagnostic', 'mae_s': value['mae_s'], 'n': value['n']})
display(pd.DataFrame(rows))
display(pd.read_csv(root / 'submission.csv', sep=';').head())


,set,mae_s,n
0,chronological_audit,66.401189,473
1,exposed_test_diagnostic,67.207461,353


,sample_id,prediction
0,131672_1767670500,62.784103
1,131672_1767670800,106.607169
2,131672_1767671100,183.439076
3,131672_1767671400,156.290612
4,134040_1767672000,48.225934


## Takeaways
Локальная проверка подтверждает файл и воспроизводимый путь, но не независимое качество: 382/473 audit-точек уже видели прежние проверки. C1 не получает внешний балл без разрешённой загрузки и platform readback. Реконструируемый validate proxy исключён из оценки.
